# Notebook 01 mini - AmazonLike prototype intersection

Questo notebook crea una versione piccola,varia e bilanciata del dataset Amazon-like per alimentare il sito prototipo.

Interseca tre sorgenti:

- metadata Amazon Reviews 2023;
- QA storico McAuley;
- rating/reviews Amazon Reviews 2023.

Output principali:

```text
prototype/data/processed/prototype_amazon_like_intersection/
  items.parquet
  qa.parquet
  ratings.parquet
  category_summary.csv
  summary.json

prototype/data/products_demo.json
```

Non modifica i notebook originali e non scrive in `data/raw/` o `data/processed/`, salvo lettura dei QA storici gia presenti/scaricabili.

## Nota esecuzione

Questo notebook usa la root assoluta del repository:

```text
C:\Users\arsid\Desktop\Tesi-ecommerce
```

Quindi puo essere eseguito anche dalla cartella `prototype/notebooks/` senza creare output annidati nella posizione sbagliata.


In [1]:
#se mancano pacchetti,esegui questa cella una sola volta
%pip install -q "datasets==3.6.0" pandas pyarrow requests tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import ast
import gzip
import json
import math
from typing import Iterable,Optional

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from tqdm.auto import tqdm
import datasets
from datasets import load_dataset

#riduciamo i log del pacchetto datasets
datasets.logging.set_verbosity_error()

c:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#configurazione dataset mini per il prototipo
DATASET_NAME="prototype_amazon_like_intersection"
DATASET_LABEL="AmazonLikePrototypeMini"

#categorie Amazon-like richieste per il prototipo
CATEGORIES=[
    {"label":"Appliances","hf_category":"Appliances","qa_slug":"Appliances","group":"home"},
    {"label":"Arts Crafts and Sewing","hf_category":"Arts_Crafts_and_Sewing","qa_slug":"Arts_Crafts_and_Sewing","group":"crafts"},
    {"label":"Automotive","hf_category":"Automotive","qa_slug":"Automotive","group":"automotive"},
    {"label":"Baby Products","hf_category":"Baby_Products","qa_slug":"Baby","group":"baby"},
    {"label":"Beauty","hf_category":"All_Beauty","qa_slug":"Beauty","group":"beauty"},
    {"label":"Cell Phones and Accessories","hf_category":"Cell_Phones_and_Accessories","qa_slug":"Cell_Phones_and_Accessories","group":"tech"},
    {"label":"Clothing Shoes and Jewelry","hf_category":"Clothing_Shoes_and_Jewelry","qa_slug":"Clothing_Shoes_and_Jewelry","group":"fashion"},
    {"label":"Electronics","hf_category":"Electronics","qa_slug":"Electronics","group":"tech"},
    {"label":"Grocery and Gourmet Food","hf_category":"Grocery_and_Gourmet_Food","qa_slug":"Grocery_and_Gourmet_Food","group":"grocery"},
    {"label":"Health and Personal Care","hf_category":"Health_and_Personal_Care","qa_slug":"Health_and_Personal_Care","group":"health"},
    {"label":"Home and Kitchen","hf_category":"Home_and_Kitchen","qa_slug":"Home_and_Kitchen","group":"home"},
    {"label":"Industrial and Scientific","hf_category":"Industrial_and_Scientific","qa_slug":"Industrial_and_Scientific","group":"industrial"},
    {"label":"Musical Instruments","hf_category":"Musical_Instruments","qa_slug":"Musical_Instruments","group":"music"},
    {"label":"Office Products","hf_category":"Office_Products","qa_slug":"Office_Products","group":"office"},
    {"label":"Patio Lawn and Garden","hf_category":"Patio_Lawn_and_Garden","qa_slug":"Patio_Lawn_and_Garden","group":"garden"},
    {"label":"Pet Supplies","hf_category":"Pet_Supplies","qa_slug":"Pet_Supplies","group":"pets"},
    {"label":"Software","hf_category":"Software","qa_slug":"Software","group":"tech"},
    {"label":"Sports and Outdoors","hf_category":"Sports_and_Outdoors","qa_slug":"Sports_and_Outdoors","group":"sports"},
    {"label":"Tools and Home Improvement","hf_category":"Tools_and_Home_Improvement","qa_slug":"Tools_and_Home_Improvement","group":"tools"},
    {"label":"Toys and Games","hf_category":"Toys_and_Games","qa_slug":"Toys_and_Games","group":"toys"},
    {"label":"Video Games","hf_category":"Video_Games","qa_slug":"Video_Games","group":"gaming"},
]

#nota mapping Beauty
#se Beauty produce 0 prodotti o problemi,provare hf_category="Beauty_and_Personal_Care" come alternativa.

QA_BASE_URL="https://mcauleylab.ucsd.edu/public_datasets/data/amazon/qa"
HF_DATASET_BASE_URL="https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main"
SAVE_TEXT_OF_REVIEW=True
FORCE_REBUILD_RATINGS=False

In [4]:
#limiti bassi per tenere il dataset adatto al prototipo
MAX_ITEMS_BY_CATEGORY={
    "Electronics":100,
    "Cell Phones and Accessories":100,
    "Home and Kitchen":100,
    "Sports and Outdoors":100,
    "Appliances":50,
    "Arts Crafts and Sewing":50,
    "Automotive":50,
    "Baby Products":50,
    "Beauty":50,
    "Clothing Shoes and Jewelry":50,
    "Grocery and Gourmet Food":50,
    "Health and Personal Care":50,
    "Industrial and Scientific":50,
    "Musical Instruments":50,
    "Office Products":50,
    "Patio Lawn and Garden":50,
    "Pet Supplies":50,
    "Software":50,
    "Tools and Home Improvement":50,
    "Toys and Games":50,
    "Video Games":50,
}

#limiti di scansione metadata Amazon Reviews 2023
MAX_METADATA_SCAN_BY_CATEGORY={
    "Electronics":150000,
    "Cell Phones and Accessories":150000,
    "Home and Kitchen":150000,
    "Sports and Outdoors":150000,
    "Appliances":80000,
    "Arts Crafts and Sewing":80000,
    "Automotive":80000,
    "Baby Products":80000,
    "Beauty":80000,
    "Clothing Shoes and Jewelry":120000,
    "Grocery and Gourmet Food":80000,
    "Health and Personal Care":80000,
    "Industrial and Scientific":80000,
    "Musical Instruments":80000,
    "Office Products":80000,
    "Patio Lawn and Garden":80000,
    "Pet Supplies":80000,
    "Software":80000,
    "Tools and Home Improvement":80000,
    "Toys and Games":80000,
    "Video Games":80000,
}

#limiti di scansione review Amazon Reviews 2023
MAX_REVIEWS_SCAN_BY_CATEGORY={
    "Electronics":150000,
    "Cell Phones and Accessories":150000,
    "Home and Kitchen":150000,
    "Sports and Outdoors":150000,
    "Appliances":80000,
    "Arts Crafts and Sewing":80000,
    "Automotive":80000,
    "Baby Products":80000,
    "Beauty":80000,
    "Clothing Shoes and Jewelry":120000,
    "Grocery and Gourmet Food":80000,
    "Health and Personal Care":80000,
    "Industrial and Scientific":80000,
    "Musical Instruments":80000,
    "Office Products":80000,
    "Patio Lawn and Garden":80000,
    "Pet Supplies":80000,
    "Software":80000,
    "Tools and Home Improvement":80000,
    "Toys and Games":80000,
    "Video Games":80000,
}

#limiti di rating salvati per categoria
MAX_RATINGS_SAVED_BY_CATEGORY={
    "Electronics":15000,
    "Cell Phones and Accessories":15000,
    "Home and Kitchen":15000,
    "Sports and Outdoors":15000,
    "Appliances":7000,
    "Arts Crafts and Sewing":7000,
    "Automotive":7000,
    "Baby Products":7000,
    "Beauty":7000,
    "Clothing Shoes and Jewelry":7000,
    "Grocery and Gourmet Food":7000,
    "Health and Personal Care":7000,
    "Industrial and Scientific":7000,
    "Musical Instruments":7000,
    "Office Products":7000,
    "Patio Lawn and Garden":7000,
    "Pet Supplies":7000,
    "Software":7000,
    "Tools and Home Improvement":7000,
    "Toys and Games":7000,
    "Video Games":7000,
}

In [5]:
#percorsi assoluti della root del repository
PROJECT_ROOT=Path(r"C:\Users\arsid\Desktop\Tesi-ecommerce")
PROTOTYPE_DIR=PROJECT_ROOT/"prototype"
RAW_DIR=PROJECT_ROOT/"data"/"raw"
QA_RAW_DIR=RAW_DIR/"qa"
QA_WORK_DIR=PROTOTYPE_DIR/"data"/"raw"/"qa"
PROCESSED_DIR=PROTOTYPE_DIR/"data"/"processed"/DATASET_NAME
PRODUCTS_DEMO_PATH=PROTOTYPE_DIR/"data"/"products_demo.json"

#percorsi output notebook
ITEMS_OUT=PROCESSED_DIR/"items.parquet"
QA_OUT=PROCESSED_DIR/"qa.parquet"
RATINGS_OUT=PROCESSED_DIR/"ratings.parquet"
CATEGORY_SUMMARY_PATH=PROCESSED_DIR/"category_summary.csv"
SUMMARY_PATH=PROCESSED_DIR/"summary.json"

#creiamo solo cartelle del prototipo
PROCESSED_DIR.mkdir(parents=True,exist_ok=True)
PRODUCTS_DEMO_PATH.parent.mkdir(parents=True,exist_ok=True)
QA_WORK_DIR.mkdir(parents=True,exist_ok=True)

print("Project root:",PROJECT_ROOT)
print("Output processed:",PROCESSED_DIR)
print("Output demo catalog:",PRODUCTS_DEMO_PATH)
print("Categorie selezionate:")
for category in CATEGORIES:
    print("-",category["label"],"| hf=",category["hf_category"],"| qa=",category["qa_slug"])

Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Output processed: C:\Users\arsid\Desktop\Tesi-ecommerce\prototype\data\processed\prototype_amazon_like_intersection
Output demo catalog: C:\Users\arsid\Desktop\Tesi-ecommerce\prototype\data\products_demo.json
Categorie selezionate:
- Appliances | hf= Appliances | qa= Appliances
- Arts Crafts and Sewing | hf= Arts_Crafts_and_Sewing | qa= Arts_Crafts_and_Sewing
- Automotive | hf= Automotive | qa= Automotive
- Baby Products | hf= Baby_Products | qa= Baby
- Beauty | hf= All_Beauty | qa= Beauty
- Cell Phones and Accessories | hf= Cell_Phones_and_Accessories | qa= Cell_Phones_and_Accessories
- Clothing Shoes and Jewelry | hf= Clothing_Shoes_and_Jewelry | qa= Clothing_Shoes_and_Jewelry
- Electronics | hf= Electronics | qa= Electronics
- Grocery and Gourmet Food | hf= Grocery_and_Gourmet_Food | qa= Grocery_and_Gourmet_Food
- Health and Personal Care | hf= Health_and_Personal_Care | qa= Health_and_Personal_Care
- Home and Kitchen | hf= Home_an

In [6]:
#scarichiamo un file se non esiste gia
def download_file(url:str,output_path:Path,force:bool=False)->Path:
    if output_path.exists() and not force:
        print(f"Gia presente: {output_path}")
        return output_path

    output_path.parent.mkdir(parents=True,exist_ok=True)
    with requests.get(url,stream=True,timeout=60) as response:
        response.raise_for_status()
        total=int(response.headers.get("content-length",0))
        with open(output_path,"wb") as file,tqdm(total=total,unit="B",unit_scale=True,desc=output_path.name) as pbar:
            for chunk in response.iter_content(chunk_size=1024*1024):
                if chunk:
                    file.write(chunk)
                    pbar.update(len(chunk))
    return output_path

#leggiamo QA storico in formato dict Python per riga
def iter_historical_qa(path:Path)->Iterable[dict]:
    with gzip.open(path,"rt",encoding="utf-8",errors="replace") as file:
        for line in file:
            line=line.strip()
            if line:
                yield ast.literal_eval(line)

#leggiamo JSONL gzip strict creato dal notebook
def iter_jsonl_gz(path:Path)->Iterable[dict]:
    with gzip.open(path,"rt",encoding="utf-8",errors="replace") as file:
        for line in file:
            line=line.strip()
            if line:
                yield json.loads(line)

#leggiamo JSONL remoto Hugging Face senza scaricarlo tutto
def iter_hf_jsonl(relative_path:str)->Iterable[dict]:
    url=f"{HF_DATASET_BASE_URL}/{relative_path}"
    with requests.get(url,stream=True,timeout=(30,300)) as response:
        response.raise_for_status()
        for line in response.iter_lines(chunk_size=1024*1024):
            if line:
                yield json.loads(line)

In [7]:
#estraiamo image_url da formati diversi del campo images
def image_url(images)->Optional[str]:
    if isinstance(images,dict):
        for key in ("hi_res","large","thumb","large_image_url","medium_image_url","small_image_url"):
            values=images.get(key) or []
            if isinstance(values,str):
                values=[values]
            for value in values:
                if value:
                    return value
    if isinstance(images,list):
        for image in images:
            if isinstance(image,dict):
                for key in ("hi_res","large","thumb","large_image_url","medium_image_url","small_image_url"):
                    value=image.get(key)
                    if value:
                        return value
    return None

#convertiamo valori complessi in stringa JSON
def tojson_string(value)->str:
    if value is None:
        return "null"
    return json.dumps(value,ensure_ascii=False)

#convertiamo valori numerici in float o None
def tofloat_or_none(value):
    try:
        if value in (None,"","None"):
            return None
        return float(value)
    except Exception:
        return None

#convertiamo valori numerici in int o None
def toint_or_none(value):
    try:
        if value in (None,"","None"):
            return None
        return int(value)
    except Exception:
        return None

#recuperiamo categoria prodotto dai metadata
def get_category(row:dict,default_category:str)->Optional[str]:
    main_category=row.get("main_category")
    if main_category not in (None,"","None"):
        return main_category

    categories=row.get("categories")
    if isinstance(categories,list) and len(categories)>0:
        return categories[0]

    return default_category

#scriviamo righe accumulate in parquet
def writerows_in_parquet(rows:list[dict],path:Path,schema:pa.Schema,writer:Optional[pq.ParquetWriter]):
    if not rows:
        return writer

    df=pd.DataFrame(rows)
    table=pa.Table.from_pandas(df,schema=schema,preserve_index=False)
    if writer is None:
        path.parent.mkdir(parents=True,exist_ok=True)
        writer=pq.ParquetWriter(path,schema=schema,compression="zstd")
    writer.write_table(table)
    return writer

#contiamo righe parquet senza caricare tutto
def parquet_count_rows(path:Path)->int:
    return pq.ParquetFile(path).metadata.num_rows

In [8]:
#convertiamo QA storico in JSONL strict e raccogliamo asin disponibili
qa_info_by_category={}
category_summary_rows=[]

for category in CATEGORIES:
    qa_slug=category["qa_slug"]
    qa_url=f"{QA_BASE_URL}/qa_{qa_slug}.json.gz"
    qa_source_path=QA_RAW_DIR/f"qa_{qa_slug}.json.gz"
    qa_cache_path=QA_WORK_DIR/f"qa_{qa_slug}.json.gz"
    qa_strict_path=QA_WORK_DIR/f"qa_{qa_slug}.strict.jsonl.gz"

    #leggiamo data/raw se il file esiste,altrimenti scarichiamo nella cartella prototype
    qa_raw_path=qa_source_path if qa_source_path.exists() else download_file(qa_url,qa_cache_path)

    qa_asins=set()
    qa_rows=0

    with gzip.open(qa_strict_path,"wt",encoding="utf-8") as output:
        for qa_id,row in enumerate(tqdm(iter_historical_qa(qa_raw_path),desc=f"Conversione QA {category['label']}")):
            asin=row.get("asin")
            row["qa_id"]=qa_id
            row["parent_asin"]=asin
            row["source_category"]=category["label"]
            row["hf_category"]=category["hf_category"]
            row["qa_slug"]=category["qa_slug"]
            row["category_group"]=category["group"]

            if asin:
                qa_asins.add(asin)

            output.write(json.dumps(row,ensure_ascii=False)+"\n")
            qa_rows+=1

    qa_info_by_category[category["label"]]={
        "qa_raw_path":qa_raw_path,
        "qa_strict_path":qa_strict_path,
        "qa_asins":qa_asins,
        "qa_rows":qa_rows,
        "qa_products":len(qa_asins),
    }
    category_summary_rows.append({
        "source_category":category["label"],
        "hf_category":category["hf_category"],
        "qa_slug":category["qa_slug"],
        "category_group":category["group"],
        "qa_rows":qa_rows,
        "qa_products":len(qa_asins),
    })

    print(category["label"],"| QA rows:",qa_rows,"| prodotti QA:",len(qa_asins))

Conversione QA Appliances: 9011it [00:00, 22510.28it/s]


Appliances | QA rows: 9011 | prodotti QA: 1153


Conversione QA Arts Crafts and Sewing: 21262it [00:00, 22989.28it/s]


Arts Crafts and Sewing | QA rows: 21262 | prodotti QA: 2944


Conversione QA Automotive: 89923it [00:03, 24500.31it/s]


Automotive | QA rows: 89923 | prodotti QA: 12536


... (32 outputs left out)

Conversione QA Toys and Games: 51486it [00:01, 26090.96it/s]


Toys and Games | QA rows: 51486 | prodotti QA: 7393


Conversione QA Video Games: 13307it [00:00, 28090.40it/s]

Video Games | QA rows: 13307 | prodotti QA: 1188


In [9]:
#schema prodotti finali
ITEM_SCHEMA=pa.schema([
    ("parent_asin",pa.string()),
    ("source_category",pa.string()),
    ("hf_category",pa.string()),
    ("qa_slug",pa.string()),
    ("category_group",pa.string()),
    ("title",pa.string()),
    ("category",pa.string()),
    ("avg_rating",pa.float64()),
    ("rating_number",pa.int64()),
    ("price",pa.float64()),
    ("store",pa.string()),
    ("features_json",pa.string()),
    ("description_json",pa.string()),
    ("categories_json",pa.string()),
    ("details_json",pa.string()),
    ("image_url",pa.string()),
])

#eliminiamo output items precedente solo nella cartella prototype
if ITEMS_OUT.exists():
    ITEMS_OUT.unlink()

writer=None
buffer=[]
global_items_seen=set()
items_id_by_category={}

#intersechiamo metadata Amazon Reviews 2023 con prodotti presenti nel QA storico
for category in CATEGORIES:
    label=category["label"]
    hf_category=category["hf_category"]
    qa_asins=qa_info_by_category[label]["qa_asins"]
    meta_path=f"raw/meta_categories/meta_{hf_category}.jsonl"

    matched_items=0
    duplicates_skipped=0
    category_items=set()
    metadata_rows_scanned=0
    max_items=MAX_ITEMS_BY_CATEGORY.get(label)
    max_metadata_scan=MAX_METADATA_SCAN_BY_CATEGORY.get(label)

    for row in tqdm(iter_hf_jsonl(meta_path),desc=f"Filtro metadati {label}"):
        if max_metadata_scan is not None and metadata_rows_scanned>=max_metadata_scan:
            break
        metadata_rows_scanned+=1

        parent_asin=row.get("parent_asin")
        if parent_asin not in qa_asins:
            continue
        if parent_asin in global_items_seen:
            duplicates_skipped+=1
            continue

        buffer.append({
            "parent_asin":parent_asin,
            "source_category":label,
            "hf_category":hf_category,
            "qa_slug":category["qa_slug"],
            "category_group":category["group"],
            "title":row.get("title"),
            "category":get_category(row,label),
            "avg_rating":tofloat_or_none(row.get("average_rating")),
            "rating_number":toint_or_none(row.get("rating_number")),
            "price":tofloat_or_none(row.get("price")),
            "store":row.get("store"),
            "features_json":tojson_string(row.get("features")),
            "description_json":tojson_string(row.get("description")),
            "categories_json":tojson_string(row.get("categories")),
            "details_json":tojson_string(row.get("details")),
            "image_url":image_url(row.get("images")),
        })

        matched_items+=1
        category_items.add(parent_asin)
        global_items_seen.add(parent_asin)

        if max_items is not None and matched_items>=max_items:
            break
        if len(buffer)>=5000:
            writer=writerows_in_parquet(buffer,ITEMS_OUT,ITEM_SCHEMA,writer)
            buffer=[]

    items_id_by_category[label]=category_items
    for summary_row in category_summary_rows:
        if summary_row["source_category"]==label:
            summary_row["metadata_rows"]=int(metadata_rows_scanned)
            summary_row["items_qa_metadata"]=int(matched_items)
            summary_row["duplicate_items_skipped"]=int(duplicates_skipped)
            break

    print(label,"| metadati scansionati:",metadata_rows_scanned,"| prodotti salvati:",matched_items,"| duplicati:",duplicates_skipped)

writer=writerows_in_parquet(buffer,ITEMS_OUT,ITEM_SCHEMA,writer)
if writer is not None:
    writer.close()

items_df=pd.read_parquet(ITEMS_OUT) if ITEMS_OUT.exists() else pd.DataFrame(columns=[field.name for field in ITEM_SCHEMA])
items_id=set(items_df["parent_asin"]) if len(items_df) else set()

print("Prodotti QA + metadata totali:",len(items_id))
display(items_df.head())

Filtro metadati Appliances: 7745it [00:01, 4231.01it/s]


Appliances | metadati scansionati: 7746 | prodotti salvati: 50 | duplicati: 0


Filtro metadati Arts Crafts and Sewing: 32530it [00:03, 9949.49it/s] 


Arts Crafts and Sewing | metadati scansionati: 32531 | prodotti salvati: 50 | duplicati: 0


Filtro metadati Automotive: 11503it [00:02, 5507.41it/s] 


Automotive | metadati scansionati: 11504 | prodotti salvati: 50 | duplicati: 0


... (33 outputs left out)

Toys and Games | metadati scansionati: 13337 | prodotti salvati: 50 | duplicati: 0


Filtro metadati Video Games: 12453it [00:03, 4038.55it/s] 

Video Games | metadati scansionati: 12454 | prodotti salvati: 50 | duplicati: 0
Prodotti QA + metadata totali: 1195


,parent_asin,source_category,hf_category,qa_slug,category_group,title,category,avg_rating,rating_number,price,store,features_json,description_json,categories_json,details_json,image_url
0,B005H47EJ4,Appliances,Appliances,Appliances,home,"GE CGS985SETSS Cafe 30"" Stainless Steel Gas Se...",Appliances,3.1,30,NaN,GE,"[""Circulates heat for high-quality results"", ""...","[""Cafe 30\"" Stainless Steel Gas Sealed Burner ...","[""Appliances"", ""Ranges, Ovens & Cooktops"", ""Ra...","{""Brand Name"": ""GE"", ""Model Info"": ""CGS985SET""...",NaN
1,B0057YO0IM,Appliances,Appliances,Appliances,home,"Bosch SHE3AR52UC Ascenta 24"" White Full Consol...",Appliances,3.0,47,NaN,BOSCH,"[""14 Place Settings"", ""4 Wash Cycles featuring...","[""24 '' Recessed Handle Dishwasher Ascenta Ser...","[""Appliances"", ""Dishwashers"", ""Built-In Dishwa...","{""Brand Name"": ""BOSCH"", ""Model Info"": ""SHE3AR5...",NaN
2,B001TIYKGC,Appliances,Appliances,Appliances,home,GARP 5300622034 Compatible Replacement for Dry...,Tools & Home Improvement,4.5,657,17.5,GARP,"[""Researched and Developed in New Jersey"", ""Li...","[""Product Description"", ""GARP products are res...","[""Appliances"", ""Parts & Accessories"", ""Dryer P...","{""Manufacturer"": ""GARP"", ""Part Number"": ""53006...",NaN
3,B004OLUEZ2,Appliances,Appliances,Appliances,home,"Empava 30"" Built-in Electric Induction Cooktop...",Appliances,4.3,117,NaN,Whirlpool,"[""The USA & Canada ETL Certified Electric Stov...","[""The 30-in 4 elements black induction cooktop...","[""Appliances"", ""Ranges, Ovens & Cooktops"", ""Co...","{""Brand Name"": ""Whirlpool"", ""Model Info"": ""30\...",NaN
4,B0002GTTRC,Appliances,Appliances,Appliances,home,GE GSWF Refrigerator Water Filter | Certified ...,Tools & Home Improvement,4.6,2030,40.0,General Electric,"[""Premium filtration: Certified to reduce part...","[""GSWF Features: -Water filter. -GE GSWF Refri...","[""Appliances"", ""Parts & Accessories"", ""Refrige...","{""Material"": ""water filter"", ""Product Dimensio...",NaN


In [10]:
#schema QA filtrate
QA_SCHEMA=pa.schema([
    ("qa_id",pa.int64()),
    ("parent_asin",pa.string()),
    ("source_category",pa.string()),
    ("hf_category",pa.string()),
    ("qa_slug",pa.string()),
    ("category_group",pa.string()),
    ("question",pa.string()),
    ("answer",pa.string()),
    ("questionType",pa.string()),
    ("answerType",pa.string()),
    ("answerTime",pa.string()),
    ("unixTime",pa.int64()),
])

#eliminiamo output QA precedente solo nella cartella prototype
if QA_OUT.exists():
    QA_OUT.unlink()

writer=None
buffer=[]
matched_qa_total=0

#teniamo solo QA dei prodotti presenti in items.parquet
for category in CATEGORIES:
    label=category["label"]
    category_items=items_id_by_category.get(label,set())
    qa_strict_path=qa_info_by_category[label]["qa_strict_path"]
    matched_qa=0

    for row in tqdm(iter_jsonl_gz(qa_strict_path),desc=f"Filtro QA {label}"):
        parent_asin=row.get("parent_asin")
        if parent_asin not in category_items:
            continue

        buffer.append({
            "qa_id":int(row.get("qa_id")),
            "parent_asin":parent_asin,
            "source_category":label,
            "hf_category":category["hf_category"],
            "qa_slug":category["qa_slug"],
            "category_group":category["group"],
            "question":row.get("question"),
            "answer":row.get("answer"),
            "questionType":row.get("questionType"),
            "answerType":row.get("answerType"),
            "answerTime":row.get("answerTime"),
            "unixTime":toint_or_none(row.get("unixTime")),
        })
        matched_qa+=1
        matched_qa_total+=1

        if len(buffer)>=10000:
            writer=writerows_in_parquet(buffer,QA_OUT,QA_SCHEMA,writer)
            buffer=[]

    for summary_row in category_summary_rows:
        if summary_row["source_category"]==label:
            summary_row["qa_rows_after_metadata"]=int(matched_qa)
            summary_row["qa_products_after_metadata"]=int(len(category_items))
            break
    print(label,"| QA filtrate:",matched_qa)

writer=writerows_in_parquet(buffer,QA_OUT,QA_SCHEMA,writer)
if writer is not None:
    writer.close()

qa_df=pd.read_parquet(QA_OUT) if QA_OUT.exists() else pd.DataFrame(columns=[field.name for field in QA_SCHEMA])
print("QA filtrate totali:",len(qa_df))
display(qa_df.head())

Filtro QA Appliances: 9011it [00:00, 202319.35it/s]


Appliances | QA filtrate: 404


Filtro QA Arts Crafts and Sewing: 21262it [00:00, 201997.55it/s]


Arts Crafts and Sewing | QA filtrate: 346


Filtro QA Automotive: 89923it [00:00, 240820.35it/s]


Automotive | QA filtrate: 358


... (33 outputs left out)

Toys and Games | QA filtrate: 357


Filtro QA Video Games: 13307it [00:00, 244341.05it/s]


Video Games | QA filtrate: 595
QA filtrate totali: 9212


,qa_id,parent_asin,source_category,hf_category,qa_slug,category_group,question,answer,questionType,answerType,answerTime,unixTime
0,137,B0002GTTRC,Appliances,Appliances,Appliances,home,Do I need to soak the new GSWF water filter in...,The directions do not indicate soaking. I have...,open-ended,NaN,"Oct 25, 2014",1.414220e+09
1,138,B0002GTTRC,Appliances,Appliances,Appliances,home,Does this fit PFE28RSHSS? I have the GE french...,You'll have to check your manual or the filter...,yes/no,?,"Mar 10, 2015",1.425971e+09
2,139,B0002GTTRC,Appliances,Appliances,Appliances,home,how do you remove old filter,Push cover to open it. It drops down. Undscrew...,open-ended,NaN,"Mar 3, 2015",1.425370e+09
3,140,B0002GTTRC,Appliances,Appliances,Appliances,home,Does it fit PF SW2MDCA SS,"Sorry, don't know what PF SW2MDCA SS is. I can...",yes/no,?,"Jan 23, 2015",1.422000e+09
4,141,B0002GTTRC,Appliances,Appliances,Appliances,home,Will this fit indoor model numer PS8688061?,I don't believe so. Looked at the model number...,yes/no,N,"Dec 17, 2014",1.418803e+09


In [11]:
#schema rating/review filtrate
rating_fields=[
    ("user_id",pa.string()),
    ("parent_asin",pa.string()),
    ("source_category",pa.string()),
    ("hf_category",pa.string()),
    ("qa_slug",pa.string()),
    ("category_group",pa.string()),
    ("asin",pa.string()),
    ("rating",pa.float64()),
    ("timestamp",pa.int64()),
    ("verified_purchase",pa.bool_()),
]

#manteniamo anche testo recensione se abilitato
if SAVE_TEXT_OF_REVIEW:
    rating_fields.extend([
        ("review_title",pa.string()),
        ("review_text",pa.string()),
    ])

RATING_SCHEMA=pa.schema(rating_fields)

#ricreiamo ratings.parquet solo se serve
if RATINGS_OUT.is_file() and RATINGS_OUT.stat().st_size>0 and not FORCE_REBUILD_RATINGS:
    n_rows=pq.ParquetFile(RATINGS_OUT).metadata.num_rows
    print(f"ratings.parquet gia presente: {RATINGS_OUT}")
    print(f"Righe gia presenti: {n_rows:,}")
else:
    if RATINGS_OUT.exists():
        RATINGS_OUT.unlink()

    writer=None
    buffer=[]
    scanned_total=0
    matched_ratings_total=0

    #scansioniamo review Amazon Reviews 2023 e salviamo solo prodotti finali candidati
    for category in CATEGORIES:
        label=category["label"]
        hf_category=category["hf_category"]
        category_items=items_id_by_category.get(label,set())
        max_scan=MAX_REVIEWS_SCAN_BY_CATEGORY.get(label)
        max_saved=MAX_RATINGS_SAVED_BY_CATEGORY.get(label)

        review_stream=load_dataset(
            "McAuley-Lab/Amazon-Reviews-2023",
            f"raw_review_{hf_category}",
            split="full",
            streaming=True,
            trust_remote_code=True,
        )

        matched_ratings=0
        scanned=0

        for row in tqdm(review_stream,desc=f"Filtro rating {label}"):
            if max_saved is not None and matched_ratings>=max_saved:
                break
            if max_scan is not None and scanned>=max_scan:
                break
            scanned+=1

            parent_asin=row.get("parent_asin")
            if parent_asin not in category_items:
                continue

            out_row={
                "user_id":row.get("user_id"),
                "parent_asin":parent_asin,
                "source_category":label,
                "hf_category":hf_category,
                "qa_slug":category["qa_slug"],
                "category_group":category["group"],
                "asin":row.get("asin"),
                "rating":tofloat_or_none(row.get("rating")),
                "timestamp":toint_or_none(row.get("timestamp")),
                "verified_purchase":row.get("verified_purchase"),
            }

            if SAVE_TEXT_OF_REVIEW:
                out_row["review_title"]=row.get("title")
                out_row["review_text"]=row.get("text")

            buffer.append(out_row)
            matched_ratings+=1
            matched_ratings_total+=1

            if len(buffer)>=20000:
                writer=writerows_in_parquet(buffer,RATINGS_OUT,RATING_SCHEMA,writer)
                buffer=[]

        scanned_total+=scanned
        for summary_row in category_summary_rows:
            if summary_row["source_category"]==label:
                summary_row["reviews_scanned"]=int(scanned)
                summary_row["ratings_saved"]=int(matched_ratings)
                break
        print(label,"| review scansionate:",scanned,"| rating salvati:",matched_ratings)

    writer=writerows_in_parquet(buffer,RATINGS_OUT,RATING_SCHEMA,writer)
    if writer is not None:
        writer.close()

    print("Review scansionate totali:",scanned_total)
    print("Rating salvati totali:",matched_ratings_total)

Filtro rating Appliances: 80000it [00:03, 21839.96it/s]


Appliances | review scansionate: 80000 | rating salvati: 483


Filtro rating Arts Crafts and Sewing: 80000it [00:03, 23086.82it/s]


Arts Crafts and Sewing | review scansionate: 80000 | rating salvati: 50


Filtro rating Automotive: 80000it [00:03, 24490.71it/s]


Automotive | review scansionate: 80000 | rating salvati: 21


... (32 outputs left out)

Filtro rating Toys and Games: 80000it [00:03, 21361.16it/s]


Toys and Games | review scansionate: 80000 | rating salvati: 32


Filtro rating Video Games: 80000it [00:03, 20407.98it/s]

Video Games | review scansionate: 80000 | rating salvati: 563
Review scansionate totali: 2000000
Rating salvati totali: 4662


In [12]:
#allineiamo items e QA ai soli prodotti con rating/review
ratings_df=pd.read_parquet(RATINGS_OUT,columns=["parent_asin","source_category"]) if RATINGS_OUT.exists() else pd.DataFrame(columns=["parent_asin","source_category"])
rating_products=set(ratings_df["parent_asin"])

items_df=items_df[items_df["parent_asin"].isin(rating_products)].copy()
items_df.to_parquet(ITEMS_OUT,index=False)
items_id=set(items_df["parent_asin"])

valid_products=set(items_df["parent_asin"])
qa_df=qa_df[qa_df["parent_asin"].isin(valid_products)].copy()
qa_df.to_parquet(QA_OUT,index=False)

#calcoliamo conteggi finali per categoria mantenendo anche categorie a zero
category_final_counts=(
    items_df.groupby("source_category")["parent_asin"]
    .nunique()
    .rename("final_products_qa_metadata_rating")
    .reset_index()
)

for summary_row in category_summary_rows:
    label=summary_row["source_category"]
    final_count=category_final_counts.loc[
        category_final_counts["source_category"]==label,
        "final_products_qa_metadata_rating",
    ]
    summary_row["final_products_qa_metadata_rating"]=int(final_count.iloc[0]) if len(final_count) else 0

category_summary_df=pd.DataFrame(category_summary_rows)
category_summary_df.to_csv(CATEGORY_SUMMARY_PATH,index=False)

print("Prodotti finali con metadata + QA + rating:",len(items_df))
print("Prodotti con QA dopo filtro:",qa_df["parent_asin"].nunique() if len(qa_df) else 0)
print("Righe QA dopo filtro:",len(qa_df))
print("Riepilogo per categoria:",CATEGORY_SUMMARY_PATH)
display(category_summary_df)

Prodotti finali con metadata + QA + rating: 491
Prodotti con QA dopo filtro: 491
Righe QA dopo filtro: 4219
Riepilogo per categoria: C:\Users\arsid\Desktop\Tesi-ecommerce\prototype\data\processed\prototype_amazon_like_intersection\category_summary.csv


,source_category,hf_category,qa_slug,category_group,qa_rows,qa_products,metadata_rows,items_qa_metadata,duplicate_items_skipped,qa_rows_after_metadata,qa_products_after_metadata,reviews_scanned,ratings_saved,final_products_qa_metadata_rating
0,Appliances,Appliances,Appliances,home,9011,1153,7746,50,0,404,50,80000,483,31
1,Arts Crafts and Sewing,Arts_Crafts_and_Sewing,Arts_Crafts_and_Sewing,crafts,21262,2944,32531,50,0,346,50,80000,50,18
2,Automotive,Automotive,Automotive,automotive,89923,12536,11504,50,0,358,50,80000,21,14
3,Baby Products,Baby_Products,Baby,baby,28933,3783,7762,50,0,385,50,80000,196,26
4,Beauty,All_Beauty,Beauty,beauty,42422,5906,80000,37,0,242,37,80000,484,33
5,Cell Phones and Accessories,Cell_Phones_and_Accessories,Cell_Phones_and_Accessories,tech,85865,10407,32275,100,0,812,100,150000,88,33
6,Clothing Shoes and Jewelry,Clothing_Shoes_and_Jewelry,Clothing_Shoes_and_Jewelry,fashion,22068,2899,120000,8,0,57,8,120000,0,0
7,Electronics,Electronics,Electronics,tech,314263,39371,7084,100,0,751,100,150000,39,22
8,Grocery and Gourmet Food,Grocery_and_Gourmet_Food,Grocery_and_Gourmet_Food,grocery,19538,3017,21286,50,0,336,50,80000,26,13
9,Health and Personal Care,Health_and_Personal_Care,Health_and_Personal_Care,health,80496,10860,29242,50,0,355,50,80000,2134,46


In [13]:
#creiamo summary generale del dataset mini
summary={
    "dataset_name":DATASET_NAME,
    "dataset_label":DATASET_LABEL,
    "categories":[category["label"] for category in CATEGORIES],
    "items":parquet_count_rows(ITEMS_OUT) if ITEMS_OUT.exists() else 0,
    "qa":parquet_count_rows(QA_OUT) if QA_OUT.exists() else 0,
    "ratings":parquet_count_rows(RATINGS_OUT) if RATINGS_OUT.exists() else 0,
    "processed_dir":str(PROCESSED_DIR),
    "items_path":str(ITEMS_OUT),
    "qa_path":str(QA_OUT),
    "ratings_path":str(RATINGS_OUT),
    "category_summary_path":str(CATEGORY_SUMMARY_PATH),
}

SUMMARY_PATH.write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding="utf-8")
summary

{'dataset_name': 'prototype_amazon_like_intersection',
 'dataset_label': 'AmazonLikePrototypeMini',
 'categories': ['Appliances',
  'Arts Crafts and Sewing',
  'Automotive',
  'Baby Products',
  'Beauty',
  'Cell Phones and Accessories',
  'Clothing Shoes and Jewelry',
  'Electronics',
  'Grocery and Gourmet Food',
  'Health and Personal Care',
  'Home and Kitchen',
  'Industrial and Scientific',
  'Musical Instruments',
  'Office Products',
  'Patio Lawn and Garden',
  'Pet Supplies',
  'Software',
  'Sports and Outdoors',
  'Tools and Home Improvement',
  'Toys and Games',
  'Video Games'],
 'items': 491,
 'qa': 4219,
 'ratings': 4662,
 'processed_dir': 'C:\\Users\\arsid\\Desktop\\Tesi-ecommerce\\prototype\\data\\processed\\prototype_amazon_like_intersection',
 'items_path': 'C:\\Users\\arsid\\Desktop\\Tesi-ecommerce\\prototype\\data\\processed\\prototype_amazon_like_intersection\\items.parquet',
 'qa_path': 'C:\\Users\\arsid\\Desktop\\Tesi-ecommerce\\prototype\\data\\processed\\prot

In [14]:
#proviamo a leggere stringhe JSON salvate nei parquet
def parse_json_value(value):
    if value in (None,"","null","None"):
        return None
    if not isinstance(value,str):
        return value
    try:
        return json.loads(value)
    except Exception:
        return value

#convertiamo descrizioni,features o details in testo leggibile
def readable_text(*values)->str:
    for value in values:
        parsed=parse_json_value(value)
        if isinstance(parsed,list):
            parts=[str(item).strip() for item in parsed if str(item).strip()]
            if parts:
                return " ".join(parts)
        if isinstance(parsed,dict):
            parts=[]
            for key,item in parsed.items():
                if isinstance(item,(str,int,float)) and not pd.isna(item):
                    parts.append(f"{key}: {item}")
            if parts:
                return " ".join(parts)
        if isinstance(parsed,str) and parsed.strip():
            return parsed.strip()
    return ""

#convertiamo NaN pandas in None prima di scrivere JSON
def clean_json_value(value):
    if isinstance(value,dict):
        return {key:clean_json_value(item) for key,item in value.items()}
    if isinstance(value,list):
        return [clean_json_value(item) for item in value]
    try:
        if pd.isna(value):
            return None
    except (TypeError,ValueError):
        pass
    return value

#carichiamo items.parquet e prepariamo il catalogo demo del sito
items_for_demo=pd.read_parquet(ITEMS_OUT) if ITEMS_OUT.exists() else pd.DataFrame()

if len(items_for_demo):
    demo=items_for_demo.copy()
    demo["description"]=demo.apply(
        lambda row: readable_text(row.get("description_json"),row.get("features_json"),row.get("details_json")),
        axis=1,
    )
    demo["item_id"]=demo["parent_asin"]
    demo["category"]=demo["source_category"]

    if "avg_rating" in demo.columns:
        demo["average_rating"]=demo["avg_rating"]
    elif "average_rating" not in demo.columns:
        demo["average_rating"]=0.0

    if "rating_number" in demo.columns:
        demo["rating_count"]=demo["rating_number"].fillna(0).astype(int)
    elif "num_ratings" in demo.columns:
        demo["rating_count"]=demo["num_ratings"].fillna(0).astype(int)
    else:
        demo["rating_count"]=0

    max_rating_count=demo["rating_count"].max()
    if max_rating_count and max_rating_count>0:
        demo["popularity_score"]=(demo["rating_count"] / max_rating_count).round(4)
    else:
        demo["popularity_score"]=0.0

    demo=demo.sort_values(["category","popularity_score"],ascending=[True,False])
    demo_records=demo[
        [
            "item_id",
            "title",
            "category",
            "description",
            "average_rating",
            "rating_count",
            "popularity_score",
            "image_url",
        ]
    ].to_dict(orient="records")
    demo_records=clean_json_value(demo_records)
else:
    demo_records=[]

#salviamo products_demo.json senza ulteriore taglio globale
PRODUCTS_DEMO_PATH.write_text(json.dumps(demo_records,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
print("products_demo.json:",PRODUCTS_DEMO_PATH)
print("Prodotti demo:",len(demo_records))

products_demo.json: C:\Users\arsid\Desktop\Tesi-ecommerce\prototype\data\products_demo.json
Prodotti demo: 491


In [15]:
#controlli finali richiesti
final_items=pd.read_parquet(ITEMS_OUT) if ITEMS_OUT.exists() else pd.DataFrame()
final_qa=pd.read_parquet(QA_OUT) if QA_OUT.exists() else pd.DataFrame()
final_ratings_count=parquet_count_rows(RATINGS_OUT) if RATINGS_OUT.exists() else 0
final_demo=pd.read_json(PRODUCTS_DEMO_PATH) if PRODUCTS_DEMO_PATH.exists() and PRODUCTS_DEMO_PATH.stat().st_size>2 else pd.DataFrame()

print("Numero prodotti totali:",len(final_items))
print("Numero categorie finali:",final_items["source_category"].nunique() if len(final_items) else 0)
print("Numero QA:",len(final_qa))
print("Numero ratings:",final_ratings_count)
print("Percorso products_demo.json:",PRODUCTS_DEMO_PATH)

print("Prodotti per categoria:")
display(category_summary_df[["source_category","final_products_qa_metadata_rating"]])

print("Prime 10 righe catalogo demo:")
display(final_demo.head(10))

Numero prodotti totali: 491
Numero categorie finali: 20
Numero QA: 4219
Numero ratings: 4662
Percorso products_demo.json: C:\Users\arsid\Desktop\Tesi-ecommerce\prototype\data\products_demo.json
Prodotti per categoria:


,source_category,final_products_qa_metadata_rating
0,Appliances,31
1,Arts Crafts and Sewing,18
2,Automotive,14
3,Baby Products,26
4,Beauty,33
5,Cell Phones and Accessories,33
6,Clothing Shoes and Jewelry,0
7,Electronics,22
8,Grocery and Gourmet Food,13
9,Health and Personal Care,46


Prime 10 righe catalogo demo:


,item_id,title,category,description,average_rating,rating_count,popularity_score,image_url
0,B001BR4R6E,UTG Heavy Duty Web Belt - Black,AMAZON FASHION,"Heavy 2"" Black Belt with Hook-and-loop System ...",4.4,1875,0.1321,NaN
1,B002GDKPUY,Special Effects SFX Hair Color Hair Dye Atomic...,All Beauty,Color: Pink Brand: Special Effects Item Form: ...,4.2,2743,0.1932,NaN
2,B00BN9PFUW,Ancient Minerals Magnesium Lotion Goodnight ni...,All Beauty,Enhanced benefits of Ancient Minerals magnesiu...,4.3,1124,0.0792,NaN
3,B00JNI1DZQ,Eyelash Curler by Bella and Bear - Your New La...,All Beauty,BEAUTY AWARD WINNERS Our products have been fe...,4.2,980,0.0690,NaN
4,B00H1M8R3C,Best Surgical Steel Nose Stud Lot - 40 Piece S...,All Beauty,Best Surgical Steel Nose Stud Lot - 40 Piece S...,4.2,524,0.0369,NaN
5,B001ET7HJI,Earth Therapeutics Mind/Body Therapy Sleep Mas...,All Beauty,Soft & smooth. Adjustable head strap. This plu...,4.5,512,0.0361,NaN
6,B00KCTER3U,Retinol Face Moisturizer Cream Natural Facial ...,All Beauty,InstaNatural’s Youth Express Age-Defying Moist...,3.7,491,0.0346,NaN
7,B000MT1GHA,"Jean Nate Silkening Body Powder, 6 Ounce",All Beauty,"A classic since 1935, Jean Nate's silkening bo...",4.4,398,0.0280,NaN
8,B000G8LWZI,Alpha Hydrox AHA Souffle Soothing Anti-Wrinkle...,All Beauty,Renew Your Skin with Anti-Wrinkle Souffle Now ...,4.0,279,0.0197,NaN
9,B000NJJFBI,"Soft 'N Style Rubber Rod Set, 60 Piece",All Beauty,Zippered carry case includes: 2 packs of purpl...,4.1,258,0.0182,NaN
